# 02 • Premier pipeline tabulaire, PyTorch et Keras

`[MÉTA | Formation 4-024 | Niveau Application | TP 02 | Mode CPU local]`

**Objectif :** Construire un modèle simple avec une baseline et un split sans fuite.

**Temps indicatif :** TP principal réparti sur J1. Ces temps sont répartis dans le conducteur, pas additionnés hors des 18 heures.

**Prérequis :** TP 00 et 01.

**Preuves de réussite :** Split disjoint, baseline, deux API, courbes, sauvegarde rechargée.

**Sources :** R01 à R04, R23.

Les jeux métier sont synthétiques. Aucun fichier personnel ou fiscal réel ne doit être chargé. Les résultats obtenus ici ne constituent pas une validation métier.

**Mode d’emploi :** exécuter les cellules dans l’ordre. Les cellules d’exercice du cahier apprenant sont à compléter ; le corrigé contient le code et des résultats de référence sur CPU.

In [1]:
from pathlib import Path
import sys, os, json
# Chercher le kit depuis le répertoire du notebook ou celui de lancement.
HERE = Path.cwd().resolve()
TP_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "modules" / "atelier.py").exists()), None)
if TP_ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dossier 03_Travaux_pratiques du kit décompressé.")
sys.path.insert(0, str(TP_ROOT / "modules"))
os.environ.setdefault("KERAS_BACKEND", "torch")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from atelier import *
seed_all(42)
print("Moteur disponible :", torch.__version__, "| Données :", DATA)


Moteur disponible : 2.10.0+cpu | Données : /mnt/data/deep_learning_4_024/03_Travaux_pratiques/donnees


## 1. Contrat de données
2 400 dossiers entièrement artificiels. Les entrées v00 à v19 ne représentent aucun critère fiscal. La cible examen_utile n’est pas un constat de fraude. resultat_apres_examen est une fuite volontaire ; dossier_id est un identifiant. Ces deux colonnes sont exclues.

In [2]:
df,X,y=read_tabular()
print(df.head(3).to_string(index=False));print("Fréquence positive :",y.mean())
from sklearn.model_selection import train_test_split
indices=np.arange(len(y))
train_idx,reste=train_test_split(indices,test_size=.30,random_state=42,stratify=y)
val_idx,test_idx=train_test_split(reste,test_size=.5,random_state=43,stratify=y[reste])
assert not set(train_idx)&set(val_idx) and not set(train_idx)&set(test_idx)
scaler=StandardScaler().fit(X[train_idx])
Xtrain=scaler.transform(X[train_idx]).astype(np.float32);ytrain=y[train_idx]
Xval=scaler.transform(X[val_idx]).astype(np.float32);yval=y[val_idx]
print("Effectifs train / validation / test :",len(train_idx),len(val_idx),len(test_idx))

dossier_id       v00       v01      v02       v03       v04       v05       v06       v07       v08       v09       v10      v11       v12      v13      v14       v15      v16       v17      v18       v19  examen_utile  resultat_apres_examen
SYN-000000  2.454638  3.012335 0.855771 -0.566472  0.792638 -2.332321 -3.246469  0.303666 -1.066868 -0.923154 -1.216859 0.763669 -0.631994 3.361825 1.022051 -1.326522 1.656801 -2.605703 7.660755  3.065642             0                      0
SYN-000001  0.307921  0.435266 1.220727 -1.213517 -0.389582 -1.321056 -1.033513 -0.451528  0.364857 -2.205586  1.735566 2.140589 -0.823532 1.297152 0.387149  2.590030 1.923738 -1.368534 3.972600 -1.234169             0                      0
SYN-000002 -2.275494 -0.968265 1.793456 -1.862250 -0.084714 -1.354694 -2.199760 -0.138029  1.471389 -0.325384 -1.476584 0.176200  0.590379 2.369232 0.453003  1.508255 3.667948 -4.344554 0.955625 -1.146986             0                      0
Fréquence positive : 0.10125
Eff

## 2. Baselines
La référence linéaire et les arbres sont comparés sur la même validation. Le test reste fermé. Le score majoritaire seul ne détecte pas les positifs.

In [3]:
lineaire=LogisticRegression(max_iter=300).fit(Xtrain,ytrain)
arbres=HistGradientBoostingClassifier(max_iter=80,random_state=42).fit(Xtrain,ytrain)
comparaison={"logistique":binary_metrics(yval,lineaire.predict_proba(Xval)[:,1]),"arbres":binary_metrics(yval,arbres.predict_proba(Xval)[:,1])}
print(pd.DataFrame(comparaison).T)

            seuil  exactitude  ...     brier  alertes
logistique    0.5    0.938889  ...  0.047807     21.0
arbres        0.5    0.961111  ...  0.037062     25.0

[2 rows x 9 columns]


## 3. Réseau PyTorch et boucle entièrement visible
La sortie est un logit. BCEWithLogitsLoss combine le traitement approprié du logit et la perte binaire. La sigmoïde est utilisée pour produire les scores à l’évaluation. La validation ne reçoit aucune mise à jour.

In [4]:
from torch.utils.data import TensorDataset,DataLoader
seed_all()
modele=nn.Sequential(nn.Linear(20,32),nn.ReLU(),nn.Linear(32,16),nn.ReLU(),nn.Linear(16,1))
optimiseur=torch.optim.Adam(modele.parameters(),lr=.003)
critere=nn.BCEWithLogitsLoss()
loader=DataLoader(TensorDataset(torch.tensor(Xtrain),torch.tensor(ytrain,dtype=torch.float32)),batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(42))
historique=[]
for epoque in range(12):
    modele.train();total=0.
    for xb,yb in loader:
        optimiseur.zero_grad(set_to_none=True)
        logits=modele(xb).squeeze(-1)
        perte=critere(logits,yb)
        perte.backward();optimiseur.step()
        total+=perte.item()*len(xb)
    modele.eval()
    with torch.no_grad():
        lv=critere(modele(torch.tensor(Xval)).squeeze(-1),torch.tensor(yval,dtype=torch.float32)).item()
    historique.append({"epoque":epoque+1,"perte_train":total/len(Xtrain),"perte_validation":lv})
historique=pd.DataFrame(historique)
plot_history(historique,"Premier réseau PyTorch","02_pytorch.png")
with torch.no_grad():scores=torch.sigmoid(modele(torch.tensor(Xval)).squeeze(-1)).numpy()
comparaison["PyTorch"]=binary_metrics(yval,scores)

## 4. Le même problème avec Keras
Keras réduit le code de boucle avec compile et fit. Le moteur actif est imprimé. Cette expérience montre deux niveaux d’API, pas une comparaison de vitesse scientifiquement contrôlée. Les initialisations et détails d’optimisation ne sont pas identiques.

In [5]:
import keras
keras.utils.set_random_seed(42)
km=keras.Sequential([keras.layers.Input((20,)),keras.layers.Dense(32,activation="relu"),keras.layers.Dense(16,activation="relu"),keras.layers.Dense(1,activation="sigmoid")])
km.compile(optimizer=keras.optimizers.Adam(learning_rate=.003),loss="binary_crossentropy",metrics=["accuracy"])
kh=km.fit(Xtrain,ytrain,validation_data=(Xval,yval),epochs=8,batch_size=64,verbose=0)
ks=km.predict(Xval,verbose=0).ravel()
comparaison["Keras_"+keras.backend.backend()]=binary_metrics(yval,ks)
print(pd.DataFrame(comparaison).T.round(4))
km.save(RESULTS/"02_modele.keras")
recharge=keras.models.load_model(RESULTS/"02_modele.keras")
np.testing.assert_allclose(ks[:10],recharge.predict(Xval[:10],verbose=0).ravel(),rtol=1e-5,atol=1e-6)
print("Rechargement Keras vérifié")

             seuil  exactitude  ...   brier  alertes
logistique     0.5      0.9389  ...  0.0478     21.0
arbres         0.5      0.9611  ...  0.0371     25.0
PyTorch        0.5      0.9583  ...  0.0318     26.0
Keras_torch    0.5      0.9556  ...  0.0368     33.0

[4 rows x 9 columns]
Rechargement Keras vérifié


## 5. Variante native TensorFlow
Cette cellule n’est exécutée que si TensorFlow est installé. Son statut est écrit dans le rapport. Elle sert à reconnaître GradientTape et apply_gradients, pas à multiplier artificiellement les exercices.

In [6]:
import importlib.util
statut_tf="non exécuté, TensorFlow absent"
if importlib.util.find_spec("tensorflow"):
    import tensorflow as tf
    tf.random.set_seed(42)
    tm=tf.keras.Sequential([tf.keras.layers.Input((20,)),tf.keras.layers.Dense(16,activation="relu"),tf.keras.layers.Dense(1)])
    opt=tf.keras.optimizers.Adam(.003)
    tx=tf.convert_to_tensor(Xtrain[:64]);ty=tf.convert_to_tensor(ytrain[:64,None],dtype=tf.float32)
    with tf.GradientTape() as tape:
        logits=tm(tx,training=True)
        tf_loss=tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=ty,logits=logits))
    gradients=tape.gradient(tf_loss,tm.trainable_variables)
    opt.apply_gradients(zip(gradients,tm.trainable_variables))
    statut_tf=f"une étape exécutée, perte {float(tf_loss.numpy()):.4f}"
print("Branche TensorFlow :",statut_tf)

Branche TensorFlow : non exécuté, TensorFlow absent


## 6. Sauvegarde PyTorch et fiche d’expérience
La comparaison porte sur les sorties d’un nouvel objet modèle. Le fichier du scaler doit rester associé aux poids. Ici, on conserve ses paramètres sous forme numérique pour le prototype. Ne charger que des fichiers maîtrisés.

In [7]:
torch.save(modele.state_dict(),RESULTS/"02_mlp.pt")
nouveau=nn.Sequential(nn.Linear(20,32),nn.ReLU(),nn.Linear(32,16),nn.ReLU(),nn.Linear(16,1))
nouveau.load_state_dict(torch.load(RESULTS/"02_mlp.pt",weights_only=True));nouveau.eval()
with torch.no_grad():np.testing.assert_allclose(modele(torch.tensor(Xval[:10])).numpy(),nouveau(torch.tensor(Xval[:10])).numpy(),atol=1e-6)
np.savez(RESULTS/"02_pretraitement.npz",mean=scaler.mean_,scale=scaler.scale_)
save_result("02_comparaison",{"validation":comparaison,"tensorflow":statut_tf,"test_utilise":False})

PosixPath('/mnt/data/deep_learning_4_024/03_Travaux_pratiques/resultats/02_comparaison.json')

## 7. Interpréter sans surpromettre
Rédiger : modèle préféré selon quelle métrique ; défaut principal ; expérience suivante. Ne pas choisir un vainqueur universel à partir de ce jeu artificiel. **Remédiation :** vérifier le split et le type de cible avant de changer la largeur du réseau. **Extension :** reprendre une expérience avec un taux différent en conservant les autres décisions.